In [ ]:

import numpy as np
from hyppo.independence import Hsic

def hsic_test(X, Y, kernel="gaussian"):
    """
    HSIC
    Kernels:
        - "gaussian"
        - "linear"
        - "poly"
        - "laplacian"
    """

    X = X.reshape(-1, 1) if X.ndim == 1 else X
    Y = Y.reshape(-1, 1) if Y.ndim == 1 else Y

    stat, pvalue = Hsic(compute_kernel=kernel).test(X, Y)

    return stat, pvalue




In [ ]:
#dep variables

np.random.seed(42)

X = np.random.normal(size=300)
Y = X**2 + 0.1 * np.random.normal(size=300)

stat, p = hsic_test(X, Y, kernel="gaussian")

print("Dependent")
print("HSIC statistic:", stat)
print("p-value:", p)

#indep

X = np.random.normal(size=300)
Y = np.random.normal(size=300)

stat, p = hsic_test(X, Y, kernel="gaussian")

print("Independent")
print("HSIC statistic:", stat)
print("p-value:", p)




Dependent
HSIC statistic: 0.41425053517047483
p-value: 4.430492261043402e-29
Independent
HSIC statistic: -0.0056654306450148505
p-value: 1.0
linear: stat=-0.00252, p=0.62212
gaussian: stat=-0.00567, p=1.00000
poly: stat=0.01652, p=0.01466
laplacian: stat=0.00251, p=0.18532


In [7]:

from scipy.spatial.distance import cdist

#kernels

def _reshape(X):
    return X.reshape(-1, 1) if X.ndim == 1 else X

def _median_bandwidth(X):
    X = _reshape(X)
    D = cdist(X, X)
    return np.median(D[D > 0])

def linear_kernel(X, Y=None):
    X = _reshape(X)
    Y = X if Y is None else _reshape(Y)
    return X @ Y.T

def rbf_kernel(X, Y=None, sigma=None):
    X = _reshape(X)
    Y = X if Y is None else _reshape(Y)

    if sigma is None:
        sigma = _median_bandwidth(X)

    D = cdist(X, Y, metric="sqeuclidean")
    return np.exp(-D / (2 * sigma**2))

def polynomial_kernel(X, Y=None, degree=3, c=1):
    X = _reshape(X)
    Y = X if Y is None else _reshape(Y)
    return (X @ Y.T + c) ** degree

def laplace_kernel(X, Y=None, sigma=None):
    X = _reshape(X)
    Y = X if Y is None else _reshape(Y)

    if sigma is None:
        sigma = _median_bandwidth(X)

    D = cdist(X, Y, metric="cityblock")
    return np.exp(-D / sigma)


KERNELS = {
    "linear": linear_kernel,
    "rbf": rbf_kernel,
    "polynomial": polynomial_kernel,
    "laplace": laplace_kernel,
}
#hsic

def hsic(X, Y, kernel_x="rbf", kernel_y="rbf", n_perm=200):

    X = _reshape(X)
    Y = _reshape(Y)

    if len(X) != len(Y):
        raise ValueError("not same nb samples")

    n = len(X)

    #kernel selection
    kernel_x = kernel_x.lower() if isinstance(kernel_x, str) else kernel_x
    kernel_y = kernel_y.lower() if isinstance(kernel_y, str) else kernel_y

    kx = KERNELS[kernel_x] if isinstance(kernel_x, str) else kernel_x
    ky = KERNELS[kernel_y] if isinstance(kernel_y, str) else kernel_y

    #kernel matrices
    Kx = kx(X)
    Ky = ky(Y)

    #centering matrix
    H = np.eye(n) - np.ones((n, n)) / n

    # Center kernels
    Kxc = H @ Kx @ H
    Kyc = H @ Ky @ H

    # HSIC statistic
    stat = np.trace(Kxc @ Kyc) / ((n - 1) ** 2)

    # Permutation test
    null = np.zeros(n_perm)

    for b in range(n_perm):
        idx = np.random.permutation(n)

        Ky_perm = Ky[idx][:, idx]
        Kyc_perm = H @ Ky_perm @ H

        null[b] = np.trace(Kxc @ Kyc_perm) / ((n - 1) ** 2)

    pvalue = np.mean(null >= stat)

    return stat, pvalue


#ex

if __name__ == "__main__":

    np.random.seed(42)

    X = np.random.normal(size=300)
    Y = X**2 + 0.1 * np.random.normal(size=300)

    for kernel in ["linear", "rbf", "polynomial", "laplace"]:

        stat, p = hsic(X, Y,
                       kernel_x=kernel,
                       kernel_y=kernel)

        print(f"{kernel:10s} HSIC={stat:.5f} p={p:.5f}")



linear     HSIC=0.02290 p=0.04500
rbf        HSIC=0.02921 p=0.00000
polynomial HSIC=416327.59913 p=0.00000
laplace    HSIC=0.02578 p=0.00000
